# Module 33: Model Interpretability — See What Your Model Sees

**What you'll learn:**
- How to use forward hooks, backward hooks, and tensor hooks
- Extracting intermediate activations without modifying model code
- Computing saliency maps (gradient-based pixel attribution)
- Implementing Grad-CAM for spatial localization
- Extracting attention weights from Transformers
- Detecting dead neurons and vanishing gradients with hooks

**Prerequisites:** Modules 04 (Neural Networks), 07 (Training Pipelines)  
**Time:** ~90 minutes

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print(f"PyTorch {torch.__version__}")

## 1. What Are Hooks?

Hooks are callbacks registered on modules or tensors that fire during forward/backward passes. They let you **observe and modify** activations and gradients without touching the model source code.

| Hook Type | Registration | Signature | When |
|-----------|-------------|-----------|------|
| Forward hook | `module.register_forward_hook(fn)` | `fn(module, input, output)` | After `forward()` |
| Forward pre-hook | `module.register_forward_pre_hook(fn)` | `fn(module, input)` | Before `forward()` |
| Backward hook | `module.register_full_backward_hook(fn)` | `fn(module, grad_input, grad_output)` | During `backward()` |
| Tensor hook | `tensor.register_hook(fn)` | `fn(grad)` | When gradient computed |

## 2. Forward Hook — Extract Activations

The most common use case: capture what a layer produces without modifying the model.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.fc3(x)

model = SimpleMLP()
x = torch.randn(4, 64)

In [ ]:
activations = {}

def save_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

handles = []
for name, module in model.named_modules():
    if name:  # skip root module
        handles.append(module.register_forward_hook(save_activation(name)))

output = model(x)

print("Captured activations:")
for name, act in activations.items():
    print(f"  {name:8s} -> shape={act.shape}, mean={act.mean():.4f}, std={act.std():.4f}")

# Clean up
for h in handles:
    h.remove()

## 3. Activation Statistics — Dead Neurons and Distribution

A key diagnostic: what fraction of ReLU outputs are zero ("dead neurons")?

In [ ]:
class ActivationStats:
    def __init__(self, model):
        self.stats = {}
        self._handles = []
        for name, module in model.named_modules():
            if isinstance(module, (nn.ReLU, nn.GELU, nn.SiLU)):
                handle = module.register_forward_hook(self._hook(name))
                self._handles.append(handle)

    def _hook(self, name):
        def fn(module, input, output):
            with torch.no_grad():
                flat = output.flatten()
                self.stats[name] = {
                    'mean': flat.mean().item(),
                    'std': flat.std().item(),
                    'dead_pct': (flat == 0).float().mean().item() * 100,
                }
        return fn

    def close(self):
        for h in self._handles:
            h.remove()

In [ ]:
model = SimpleMLP()
x = torch.randn(32, 64)

# Normal initialization
stats = ActivationStats(model)
_ = model(x)
print("Normal initialization:")
for name, s in stats.stats.items():
    print(f"  {name}: mean={s['mean']:.4f}, std={s['std']:.4f}, dead={s['dead_pct']:.1f}%")
stats.close()

# Bad initialization — forces many dead neurons
model.fc1.bias.data.fill_(-5.0)
stats2 = ActivationStats(model)
_ = model(x)
print("\nBad initialization (bias=-5):")
for name, s in stats2.stats.items():
    print(f"  {name}: mean={s['mean']:.4f}, std={s['std']:.4f}, dead={s['dead_pct']:.1f}%")
stats2.close()

## 4. Backward Hook — Gradient Norms

Monitor gradient flow through the network during backpropagation.

In [ ]:
model = SimpleMLP()
x = torch.randn(4, 64)
target = torch.randint(0, 10, (4,))

grad_norms = {}

def log_grad_norm(name):
    def hook(module, grad_input, grad_output):
        grad_norms[name] = grad_output[0].norm().item()
    return hook

handles = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Linear, nn.ReLU)):
        handles.append(module.register_full_backward_hook(log_grad_norm(name)))

loss = F.cross_entropy(model(x), target)
loss.backward()

print(f"Loss: {loss.item():.4f}")
print("\nGradient norms (backward direction):")
for name, norm in grad_norms.items():
    bar = '#' * int(norm * 20)
    print(f"  {name:8s}: {norm:.6f}  {bar}")

for h in handles:
    h.remove()

## 5. Saliency Maps — Which Pixels Matter?

The simplest gradient-based attribution: compute `|d(score)/d(input)|`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.relu3 = nn.ReLU()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.relu3(self.conv3(x))
        x = self.gap(x).flatten(1)
        return self.fc(x)

cnn = SimpleCNN()
image = torch.randn(1, 3, 32, 32)

In [ ]:
def compute_saliency(model, input_tensor, target_class):
    model.eval()
    inp = input_tensor.clone().requires_grad_(True)
    output = model(inp)
    score = output[0, target_class]
    model.zero_grad()
    score.backward()

    saliency = inp.grad.abs().squeeze(0)
    if saliency.dim() == 3:
        saliency = saliency.max(dim=0).values
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency

pred_class = cnn(image).argmax(dim=1).item()
saliency = compute_saliency(cnn, image, pred_class)

print(f"Predicted class: {pred_class}")
print(f"Saliency shape: {saliency.shape}")
print(f"High-saliency pixels (>0.5): {(saliency > 0.5).sum().item()}/{saliency.numel()}")

# Text-based heatmap
ds = F.adaptive_avg_pool2d(saliency.unsqueeze(0).unsqueeze(0), 8).squeeze()
print("\nSaliency heatmap (8x8, 0-9 scale):")
for r in range(8):
    print("  " + " ".join(str(int(ds[r, c].item() * 9)) for c in range(8)))

## 6. Grad-CAM — Where Does the Model Look?

Grad-CAM produces a coarse heatmap showing which spatial regions a CNN focuses on.

**Algorithm:**
1. Hook the last conv layer to capture activations and gradients
2. Forward pass → backward from target class
3. Global-average-pool gradients → channel weights
4. Weighted sum of activation maps → ReLU → upsample

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._handles = [
            target_layer.register_forward_hook(self._save_act),
            target_layer.register_full_backward_hook(self._save_grad),
        ]

    def _save_act(self, module, input, output):
        self.activations = output.detach()

    def _save_grad(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        return cam.squeeze()

    def close(self):
        for h in self._handles:
            h.remove()

In [ ]:
cam = GradCAM(cnn, cnn.conv3)
heatmap = cam.generate(image, pred_class)
cam.close()

print(f"Grad-CAM heatmap shape: {heatmap.shape}")
print(f"Hot region (>0.5): {(heatmap > 0.5).float().mean().item()*100:.1f}%")

ds = F.adaptive_avg_pool2d(heatmap.unsqueeze(0).unsqueeze(0), 8).squeeze()
print("\nGrad-CAM heatmap (8x8, 0-9 scale):")
for r in range(8):
    print("  " + " ".join(str(int(ds[r, c].item() * 9)) for c in range(8)))

In [ ]:
# Compare Grad-CAM across different target classes
print("Grad-CAM for different target classes:")
for cls in range(3):
    cam = GradCAM(cnn, cnn.conv3)
    hm = cam.generate(image, cls)
    cam.close()
    hot = (hm > 0.5).float().mean().item() * 100
    print(f"  Class {cls}: mean={hm.mean():.4f}, hot region={hot:.1f}%")

## 7. Attention Weight Extraction

For Transformer models, hook into `MultiheadAttention` to see which tokens attend to which.

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, num_classes=5):
        super().__init__()
        self.embedding = nn.Linear(d_model, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        return self.classifier(x.mean(dim=1))

transformer = TinyTransformer()
seq = torch.randn(2, 8, 64)  # batch=2, seq_len=8, d_model=64

In [ ]:
class AttentionExtractor:
    def __init__(self, model):
        self.attention_maps = {}
        self._handles = []
        for name, module in model.named_modules():
            if isinstance(module, nn.MultiheadAttention):
                handle = module.register_forward_hook(self._hook(name))
                self._handles.append(handle)

    def _hook(self, name):
        def fn(module, input, output):
            if isinstance(output, tuple) and len(output) == 2 and output[1] is not None:
                self.attention_maps[name] = output[1].detach()
        return fn

    def close(self):
        for h in self._handles:
            h.remove()

extractor = AttentionExtractor(transformer)
out = transformer(seq)
extractor.close()

print(f"Captured attention from {len(extractor.attention_maps)} layers")
for name, attn in extractor.attention_maps.items():
    print(f"  {name}: shape={attn.shape}")
    if attn.dim() == 3:
        sample = attn[0]
        print(f"  Attention matrix (first 4x4 of sample 0):")
        for r in range(min(4, sample.shape[0])):
            vals = [f"{sample[r, c]:.3f}" for c in range(min(4, sample.shape[1]))]
            print(f"    [{', '.join(vals)}, ...]")

## 8. FeatureExtractor Pattern

A reusable class that registers hooks, collects activations, and cleans up properly.

In [ ]:
class FeatureExtractor:
    def __init__(self, model, target_layers):
        self.model = model
        self.features = {}
        self._handles = []
        layer_map = dict(model.named_modules())
        for name in target_layers:
            handle = layer_map[name].register_forward_hook(self._hook(name))
            self._handles.append(handle)

    def _hook(self, name):
        def fn(module, input, output):
            self.features[name] = output.detach()
        return fn

    def __call__(self, x):
        self.features.clear()
        output = self.model(x)
        return output, dict(self.features)

    def close(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

In [ ]:
model = SimpleMLP()
x = torch.randn(4, 64)

with FeatureExtractor(model, ['fc1', 'relu1', 'fc2', 'fc3']) as ext:
    output, features = ext(x)

print("Extracted features:")
for name, feat in features.items():
    print(f"  {name}: shape={feat.shape}, mean={feat.mean():.4f}")

# Verify hooks were cleaned up
hooks_left = sum(len(m._forward_hooks) for m in model.modules())
print(f"\nHooks remaining: {hooks_left} (should be 0)")

## 9. Cleanup — RemovableHandle

Three patterns for proper hook cleanup. **Always** remove hooks after use to prevent memory leaks.

In [ ]:
from contextlib import contextmanager

@contextmanager
def hook_context(module, hook_fn, hook_type='forward'):
    """Context manager for automatic hook cleanup."""
    if hook_type == 'forward':
        handle = module.register_forward_hook(hook_fn)
    elif hook_type == 'backward':
        handle = module.register_full_backward_hook(hook_fn)
    elif hook_type == 'pre':
        handle = module.register_forward_pre_hook(hook_fn)
    else:
        raise ValueError(f"Unknown hook_type: {hook_type}")
    try:
        yield handle
    finally:
        handle.remove()

In [ ]:
model = SimpleMLP()
x = torch.randn(4, 64)

# Pattern 1: Manual
captured = []
handle = model.fc1.register_forward_hook(lambda m, i, o: captured.append(o.shape))
_ = model(x)
handle.remove()
print(f"Pattern 1 (manual): {captured[0]}")

# Pattern 2: Context manager
captured2 = []
with hook_context(model.fc2, lambda m, i, o: captured2.append(o.shape)):
    _ = model(x)
print(f"Pattern 2 (context mgr): {captured2[0]}")

# Pattern 3: FeatureExtractor class
with FeatureExtractor(model, ['fc3']) as ext:
    _, feats = ext(x)
print(f"Pattern 3 (class): {feats['fc3'].shape}")

total = sum(len(m._forward_hooks) + len(m._forward_pre_hooks) + len(m._backward_hooks) for m in model.modules())
print(f"\nAll hooks cleaned up: {total == 0}")

## 10. Guided Backpropagation

Produces sharper attributions by only passing positive gradients through ReLU.

In [ ]:
class GuidedBackprop:
    def __init__(self, model):
        self.model = model
        self._handles = []
        for module in model.modules():
            if isinstance(module, nn.ReLU):
                handle = module.register_full_backward_hook(self._hook)
                self._handles.append(handle)

    @staticmethod
    def _hook(module, grad_input, grad_output):
        return (torch.clamp(grad_output[0], min=0.0),)

    def generate(self, input_tensor, target_class):
        self.model.eval()
        inp = input_tensor.clone().requires_grad_(True)
        output = self.model(inp)
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot)
        return inp.grad.clone()

    def close(self):
        for h in self._handles:
            h.remove()

In [ ]:
cnn = SimpleCNN()
image = torch.randn(1, 3, 32, 32)
pred = cnn(image).argmax(dim=1).item()

# Vanilla saliency
vanilla = compute_saliency(cnn, image, pred)

# Guided backprop
gbp = GuidedBackprop(cnn)
guided = gbp.generate(image, pred)
gbp.close()
guided_map = guided.abs().squeeze(0).max(dim=0).values
guided_map = (guided_map - guided_map.min()) / (guided_map.max() - guided_map.min() + 1e-8)

vanilla_sparse = (vanilla < 0.01).float().mean().item()
guided_sparse = (guided_map < 0.01).float().mean().item()
print(f"Sparsity (near-zero fraction):")
print(f"  Vanilla saliency: {vanilla_sparse:.4f}")
print(f"  Guided backprop:  {guided_sparse:.4f}")
print("Guided backprop produces sparser (sharper) attributions.")

## 11. Tensor Hooks — Per-Parameter Gradient Logging

In [ ]:
model = SimpleMLP()
x = torch.randn(4, 64)
target = torch.randint(0, 10, (4,))

param_grad_norms = {}
handles = []
for name, param in model.named_parameters():
    def make_hook(n):
        def hook(grad):
            param_grad_norms[n] = grad.norm().item()
        return hook
    handles.append(param.register_hook(make_hook(name)))

loss = F.cross_entropy(model(x), target)
loss.backward()

print("Per-parameter gradient norms (via tensor hooks):")
for name, norm in param_grad_norms.items():
    print(f"  {name:15s}: {norm:.6f}")

for h in handles:
    h.remove()

## 12. FLOP Counter via Hooks

In [ ]:
class FLOPCounter:
    def __init__(self, model):
        self.flops = {}
        self._handles = []
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                handle = module.register_forward_hook(self._linear_hook(name))
                self._handles.append(handle)
            elif isinstance(module, nn.Conv2d):
                handle = module.register_forward_hook(self._conv_hook(name))
                self._handles.append(handle)

    def _linear_hook(self, name):
        def hook(module, input, output):
            batch = input[0].shape[0]
            self.flops[name] = 2 * module.in_features * module.out_features * batch
        return hook

    def _conv_hook(self, name):
        def hook(module, input, output):
            batch = output.shape[0]
            oh, ow = output.shape[2], output.shape[3]
            kernel = module.kernel_size[0] * module.kernel_size[1] * (module.in_channels // module.groups)
            self.flops[name] = 2 * kernel * module.out_channels * oh * ow * batch
        return hook

    def close(self):
        for h in self._handles:
            h.remove()

model = SimpleMLP()
counter = FLOPCounter(model)
_ = model(torch.randn(8, 64))
total = sum(counter.flops.values())
print(f"{'Layer':<10} {'FLOPs':>12} {'%':>6}")
for name, f in counter.flops.items():
    print(f"{name:<10} {f:>12,} {f/total*100:>5.1f}%")
print(f"{'Total':<10} {total:>12,}")
counter.close()

## Exercise: Vanishing Gradient Detector

**Task:** Build a hook that detects layers with vanishing gradients (norm < 1e-6).

Hints:
1. Register a `register_full_backward_hook` on each `nn.Linear` layer
2. Inside the hook, compute `grad_output[0].norm()`
3. If the norm is below the threshold, record a warning
4. Test with a deep network using `nn.Sigmoid` activations

In [ ]:
# YOUR CODE HERE
# class VanishingGradientDetector:
#     def __init__(self, model, threshold=1e-6):
#         ...
#     def _check_hook(self, name):
#         ...
#     def report(self):
#         ...
#     def close(self):
#         ...

In [ ]:
# Solution
class VanishingGradientDetector:
    def __init__(self, model, threshold=1e-6):
        self.threshold = threshold
        self.grad_norms = {}
        self.warnings = []
        self._handles = []
        for name, module in model.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                handle = module.register_full_backward_hook(self._check_hook(name))
                self._handles.append(handle)

    def _check_hook(self, name):
        def hook(module, grad_input, grad_output):
            norm = grad_output[0].norm().item()
            self.grad_norms[name] = norm
            if norm < self.threshold:
                self.warnings.append(f"{name}: norm={norm:.2e}")
        return hook

    def report(self):
        for name, norm in self.grad_norms.items():
            flag = " <<< VANISHING" if norm < self.threshold else ""
            print(f"  {name:20s}: {norm:.6e}{flag}")
        if self.warnings:
            print(f"\n  {len(self.warnings)} layers with vanishing gradients!")

    def close(self):
        for h in self._handles:
            h.remove()

# Test with a deep sigmoid network
deep = nn.Sequential(*[nn.Sequential(nn.Linear(32, 32), nn.Sigmoid()) for _ in range(10)], nn.Linear(32, 5))
detector = VanishingGradientDetector(deep, threshold=1e-6)

x = torch.randn(4, 32)
loss = F.cross_entropy(deep(x), torch.randint(0, 5, (4,)))
loss.backward()

print("Gradient norms across deep sigmoid network:")
detector.report()
detector.close()

## Key Takeaways

1. **Hooks decouple observation from computation** — inspect any layer without modifying model code
2. **Forward hooks** capture activations after `forward()`; **pre-hooks** run before; **backward hooks** run during `.backward()`
3. **Tensor hooks** (`register_hook`) operate on individual tensors, useful for per-parameter gradient control
4. **Saliency maps** = `|d(score)/d(input)|` — simple but noisy pixel attribution
5. **Grad-CAM** = activation-weighted by gradient importance — coarse but semantically meaningful spatial maps
6. **Guided backpropagation** = only pass positive gradients through ReLU — sharper attributions
7. **Always remove hooks** after use (memory leak risk) — use context managers or `close()` patterns
8. **Don't modify outputs in hooks during training** — it can break the autograd graph
9. **Activation statistics** (dead neuron fraction, gradient norms) are powerful diagnostics for training health